# HK real-estate cycle and equity reaction (exploratory)

This notebook checks whether the main residential indices bottomed in 2025 and explores whether housing, credit, rental, transaction, construction, and supply variables move with listed property equities. It is deliberately diagnostic: source grain, missingness, revision semantics, and short histories are profiled before correlations are interpreted. Correlation is not a causal or investment conclusion.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'apps/asia-markets-dashboard/.generated/hk-real-estate-artifact.json').exists():
    ROOT = Path('..').resolve()
ARTIFACT = json.loads((ROOT / 'apps/asia-markets-dashboard/.generated/hk-real-estate-artifact.json').read_text())
DATASETS = ARTIFACT['snapshot']['datasets']
print('generated_at:', ARTIFACT['manifest']['generatedAt'])
print('data_as_of:', ARTIFACT['manifest']['dataAsOf'])


In [ ]:
def quality_profile(name):
    df = pd.DataFrame(DATASETS.get(name, []))
    if df.empty:
        return {'dataset': name, 'rows': 0, 'columns': '', 'duplicate_rows': 0, 'date_min': None, 'date_max': None, 'null_heavy_columns': ''}
    date_cols = [c for c in df.columns if any(token in c.lower() for token in ('date', 'month', 'period'))]
    parsed = pd.concat([pd.to_datetime(df[c], errors='coerce') for c in date_cols], ignore_index=True) if date_cols else pd.Series(dtype='datetime64[ns]')
    null_heavy = ', '.join(f'{c}={v:.0%}' for c, v in df.isna().mean().sort_values(ascending=False).items() if v >= .5)
    return {'dataset': name, 'rows': len(df), 'columns': ', '.join(df.columns), 'duplicate_rows': int(df.duplicated().sum()), 'date_min': parsed.min(), 'date_max': parsed.max(), 'null_heavy_columns': null_heavy}

profile = pd.DataFrame([quality_profile(k) for k in sorted(DATASETS)])
profile[['dataset', 'rows', 'duplicate_rows', 'date_min', 'date_max', 'null_heavy_columns']]

In [ ]:
def level(name, value='value', date='date'):
    df = pd.DataFrame(DATASETS[name]).copy()
    df[date] = pd.to_datetime(df[date], errors='coerce')
    return df.dropna(subset=[date]).set_index(date)[value].sort_index()

levels = {
    'CCL': level('ccl_history'),
    'MHPI': level('mhpi_history'),
    'RVD price': level('rvd_history', 'price'),
    'RVD rent': level('rvd_history', 'rent'),
}
epi_eri = pd.DataFrame(DATASETS['epi_eri_history'])
epi_eri['date'] = pd.to_datetime(epi_eri['date'], errors='coerce')
for label, group in epi_eri.groupby('series'):
    levels[label] = group.set_index('date')['value'].sort_index()

monthly = pd.concat(levels, axis=1).resample('ME').last()
troughs = []
for name, series in monthly.loc['2021':].items():
    s = series.dropna()
    troughs.append({'series': name, 'trough_date': s.idxmin().date(), 'trough_value': round(float(s.min()), 2), 'latest_date': s.index[-1].date(), 'trough_to_latest': round(float(s.iloc[-1] / s.min() - 1), 3)})
pd.DataFrame(troughs).sort_values('trough_date')

The trough table should be read as a timing check, not proof of a single common bottom. Weekly and monthly aggregation can move the apparent trough by several weeks; RVD is provisional and 28Hse has a different methodology.

In [ ]:
import yfinance as yf

TICKERS = ['0001.HK', '0012.HK', '0016.HK', '0017.HK', '0083.HK', '0101.HK', '0688.HK', '0823.HK', '1109.HK', '1113.HK', '1972.HK', '1997.HK', '2778.HK', '0778.HK', '0808.HK', '0435.HK', '1881.HK', '^HSI']
prices = yf.download(TICKERS, start='2020-01-01', end='2026-07-29', auto_adjust=False, progress=False, threads=False)
close = prices['Close'].resample('ME').last()
adj_close = prices['Adj Close'].resample('ME').last()
returns = adj_close.pct_change(fill_method=None)

stock_troughs = []
for ticker, series in close.loc['2021':].items():
    s = series.dropna()
    if len(s) < 10:
        continue
    stock_troughs.append({'ticker': ticker, 'raw_close_trough': s.idxmin().date(), 'raw_close_trough_value': round(float(s.min()), 2), 'raw_close_trough_to_latest': round(float(s.iloc[-1] / s.min() - 1), 3)})
pd.DataFrame(stock_troughs).sort_values('raw_close_trough')

In [ ]:
index_returns = monthly.pct_change(fill_method=None)

# k > 0 means the equity return occurs before the index return; this is only
# a screening statistic and is vulnerable to autocorrelation and common shocks.
def best_lead_lag(stock, index, lags=range(-3, 7)):
    values = []
    for k in lags:
        pair = pd.concat([returns[stock], index_returns[index].shift(-k)], axis=1).dropna().loc['2021':]
        values.append({'ticker': stock, 'index': index, 'k_months_stock_leads': k, 'corr': pair.iloc[:, 0].corr(pair.iloc[:, 1]), 'n': len(pair)})
    return pd.DataFrame(values)

screen = pd.concat([best_lead_lag(t, i) for t in ['0001.HK', '0012.HK', '0016.HK', '0017.HK', '0823.HK', '1109.HK', '1113.HK', '1972.HK', '1997.HK', '2778.HK', '0778.HK', '0808.HK', '0435.HK'] for i in ['CCL', 'MHPI', 'RVD price', 'RVD rent', 'EPI', 'ERI']], ignore_index=True)
screen.loc[screen.groupby(['ticker', 'index'])['corr'].apply(lambda s: s.abs().idxmax())].sort_values(['index', 'corr'], ascending=[True, False]).head(40)

In [ ]:
# Candidate channel summaries. These are intentionally not joined to equities
# until their grain and semantics are reviewed.
hkma = pd.DataFrame(DATASETS['hkma_mortgage_activity'])
hkma['date'] = pd.to_datetime(hkma['date'], errors='coerce')
ltv = pd.DataFrame(DATASETS['hkma_ltv_history'])
ltv['date'] = pd.to_datetime(ltv['date'], errors='coerce')
credit = pd.DataFrame(DATASETS['hkma_credit_quality_history'])
credit['date'] = pd.to_datetime(credit['date'], errors='coerce')
confidence = level('confidence_history')
print('HKMA annual mortgage activity')
display(hkma.set_index('date').loc['2020':].resample('YE')[['new_applications_count', 'approved_loans_amount_mhkd', 'approved_secondary_amount_mhkd', 'drawn_down_amount_mhkd']].sum().round())
print('HKMA annual average LTV and delinquency')
display(pd.concat({'LTV': ltv.pivot(index='date', columns='series', values='value').iloc[:, 0], 'delinquency': credit.pivot(index='date', columns='series', values='value')['Delinquency Ratio (%)']}, axis=1).loc['2020':].resample('YE').mean().round(4))
print('Midland confidence annual average')
display(confidence.loc['2020':].resample('YE').mean().round(2))
print('Potential quality flags: HKMA rate mix omits `other`; Land Registry ASP history is only 13 months; BD supply is a current snapshot; agency dedup should be checked for cross-agency matches before use.')